In [1]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# os.environ["OMP_NUM_THREADS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"                                      
os.environ["PATH"] = "/root/miniconda3/envs/vllm/bin" + os.pathsep + os.environ["PATH"]

hyperparams

In [ ]:
model_id, model_name = (
    # "Qwen/Qwen3-VL-4B-Instruct", "qwen3_vl_4b_i",
    # "Qwen/Qwen3.5-4B", "qwen3_5_4b",
    # "Qwen/Qwen3-4B-Instruct-2507", "qwen3_4b_i",
    # "Qwen/Qwen2.5-3B-Instruct", "qwen2_5_3b_i",
    # "mistralai/Mistral-7B-v0.1", "llasmol_mistral_7b",
    # "baffo32/decapoda-research-llama-7B-hf", "llama_molinst_molecule_7b",
    # "AI4Chem/ChemLLM-7B-Chat", "chemllm_7b_chat",
    # "AI4Chem/ChemLLM-7B-Chat-1_5-SFT", "chemllm_7b_chat_1_5_sft",
    # "AI4Chem/ChemLLM-7B-Chat-1_5-DPO", "chemllm_7b_chat_1_5_dpo",
    # "AI4Chem/ChemVLM-8B", "chemvlm_8b",
)

if model_name in ["llasmol_mistral_7b"]:
    lora_adapter_id = "osunlp/LlaSMol-Mistral-7B"
elif model_name in ["llama_molinst_molecule_7b"]:
    lora_adapter_id = "zjunlp/llama-molinst-molecule-7b"
else:
    print(f"Warning: No LoRA adapter found for model {model_name}. Using the base model without LoRA.")
    lora_adapter_id = None

data_id, data_name, data_split = (
    "liupf/ChEBI-20-MM", "chebi_20_mm", "test",
)

max_model_len = 4096

### Load dataset

In [3]:
from datasets import load_dataset

test_hf = load_dataset(data_id, split=data_split)

In [4]:
test_hf.column_names

['CID',
 'SMILES',
 'description',
 'polararea',
 'xlogp',
 'inchi',
 'iupacname',
 'SELFIES']

In [5]:
sample = test_hf[0]
sample

{'CID': 5354212,
 'SMILES': 'COC(=O)/C=C1/CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@]4(C)[C@H]3C(=O)C[C@]12C',
 'description': 'The molecule is a steroid ester that is methyl (17E)-pregna-4,17-dien-21-oate substituted by oxo groups at positions 3 and 11. It is a 3-oxo-Delta(4) steroid, an 11-oxo steroid, a steroid ester and a methyl ester. It derives from a hydride of a pregnane.',
 'polararea': 60.4,
 'xlogp': 2.5,
 'inchi': 'InChI=1S/C22H28O4/c1-21-9-8-15(23)10-13(21)4-6-16-17-7-5-14(11-19(25)26-3)22(17,2)12-18(24)20(16)21/h10-11,16-17,20H,4-9,12H2,1-3H3/b14-11-/t16-,17-,20+,21-,22+/m0/s1',
 'iupacname': 'methyl (2Z)-2-[(8S,9S,10R,13S,14S)-10,13-dimethyl-3,11-dioxo-1,2,6,7,8,9,12,14,15,16-decahydrocyclopenta[a]phenanthren-17-ylidene]acetate',
 'SELFIES': '[C][O][C][=Branch1][C][=O][/C][=C][/C][C][C@H1][C@@H1][C][C][C][=C][C][=Branch1][C][=O][C][C][C@][Ring1][#Branch1][Branch1][C][C][C@H1][Ring1][N][C][=Branch1][C][=O][C][C@][Ring2][Ring1][Ring2][Ring1][P][C]'}

### Launch vllm

In [13]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model=model_id,
    trust_remote_code=True,
    max_model_len=max_model_len,
    gpu_memory_utilization=0.85,
    max_num_seqs=256,
    enable_lora=True,
)

sampling_params = SamplingParams(
    temperature=.0,
    max_tokens=128,   
)

# lora_adapter = LoRARequest(
#     model_name,
#     1,
#     lora_adapter_id,
# )

INFO 08-02 02:45:13 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 256, 'disable_log_stats': True, 'enable_lora': True, 'model': 'AI4Chem/ChemVLM-8B'}


config.json:   0%|          | 0.00/5.52k [00:00<?, ?B/s]

configuration_internvl_chat.py:   0%|          | 0.00/3.94k [00:00<?, ?B/s]

configuration_internlm2.py:   0%|          | 0.00/7.00k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/AI4Chem/ChemVLM-8B:
- configuration_internlm2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

[transformers] A new version of the following files was downloaded from https://huggingface.co/AI4Chem/ChemVLM-8B:
- tokenization_internlm2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer.model:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/179 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

RuntimeError: INTERNAL: piece must not include null character.

In [8]:
rep_type, rep_type_raw_colname, rep_type_clean_colname = (
    "SMILES", "SMILES", "smiles"
    # "SELFIES", "SELFIES", "selfies"
    # "InChI", "inchi", "inchi"
    # "IUPAC name", "iupacname", "iupacname"
)

chem_property, chem_property_raw_colname, chem_property_clean_colname = (
    "XLogP", "xlogp", "xlogp",
    # "TPSA", "polararea", "tpsa",
)

In [9]:
prompt = """
Given the {rep_type}: {rep}, predict the {chem_property} value of the molecule. Do not output anything else.
""".strip()

def build_message(rep, rep_type, chem_property):
    message = [
        {
            "role": "user", 
            "content": prompt.format(rep=rep, rep_type=rep_type, chem_property=chem_property)
        },
    ]
    return message

from tqdm import tqdm

messages = []
for sample in tqdm(test_hf):
    rep = sample[rep_type_raw_colname]
    message = build_message(rep, rep_type, chem_property)
    messages.append(message)


100%|██████████| 3297/3297 [00:00<00:00, 11942.96it/s]


### Debug on one sample

In [24]:
demo_message = messages[0]
display(demo_message)

demo_prompt = demo_message[0]["content"]

# outputs = llm.chat(
#     messages=[demo_message],
#     sampling_params=sampling_params,
#     lora_request=lora_adapter,
# )

demo_prompt = """
Given the {rep_type}: <{rep_type}>{rep}</{rep_type}>, predict the {chem_property} value of the molecule.
""".strip().format(rep_type=rep_type, rep=sample[rep_type_raw_colname], chem_property=chem_property)

# demo_prompt = """
# How soluble is <SMILES> CC(C)Cl </SMILES> ?
# """.strip()

demo_prompt = """
Predict the octanol/water distribution coefficient logD under the circumstance of pH 7.4 for <SMILES> NC(=O)C1=CC=CC=C1O </SMILES>.
""".strip()

# demo_prompt = """
# Could you provide the SMILES for <IUPAC> 4-ethyl-4-methyloxolan-2-one </IUPAC> ?
# """.strip()

# demo_prompt = """
# How soluble is <SMILES> CC(C)Cl </SMILES> ?
# """.strip()

outputs = llm.generate(
    [demo_prompt],
    sampling_params=sampling_params,
    lora_request=lora_adapter,
)
raw_responses = [output.outputs[0].text for output in outputs]

output = outputs[0]
display(output.prompt)
display(len(output.prompt_token_ids))
output.outputs[0].text

[{'role': 'user',
  'content': 'Given the SMILES: COC(=O)/C=C1/CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@]4(C)[C@H]3C(=O)C[C@]12C, predict the XLogP value of the molecule. Do not output anything else.'}]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.82it/s, est. speed input: 92.36 toks/s, output: 57.26 toks/s]


'Predict the octanol/water distribution coefficient logD under the circumstance of pH 7.4 for <SMILES> NC(=O)C1=CC=CC=C1O </SMILES>.'

50

' [IUPAC] N-hydroxy-2-oxidobenzenecarboximidate </IUPAC> '

### Full inference

In [ ]:
outputs = llm.chat(
    messages=messages,
    sampling_params=sampling_params,
    lora_request=lora_adapter,
)

Rendering conversations:   0%|          | 0/3297 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 3297/3297 [00:18<00:00, 176.08it/s, est. speed input: 17608.48 toks/s, output: 1301.11 toks/s]


In [131]:
raw_responses = [output.outputs[0].text for output in outputs]

### Viz results

In [132]:
output_file_suffix = f"{data_name}_{model_name}_{rep_type_clean_colname}_{chem_property_clean_colname}"

In [133]:
import pandas as pd

raw_df = pd.DataFrame({
    "cid": list(test_hf["CID"]),
    "model_id": [model_id] * len(test_hf),
    "rep_type": [rep_type] * len(test_hf),
    "tested_chem_property": [chem_property] * len(test_hf),
    "smiles": list(test_hf["SMILES"]),
    "selfies": list(test_hf["SELFIES"]),
    "inchi": list(test_hf["inchi"]),
    "iupacname": list(test_hf["iupacname"]),
    "xlogp": list(test_hf["xlogp"]),
    "tpsa": list(test_hf["polararea"]),
    "raw_responses": raw_responses,
})
raw_df = raw_df.reset_index()
raw_df.head(2)

,index,cid,model_id,rep_type,tested_chem_property,smiles,selfies,inchi,iupacname,xlogp,tpsa,raw_responses
0,0,5354212,Qwen/Qwen3-4B-Instruct-2507,IUPAC name,XLogP,COC(=O)/C=C1/CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@]4...,[C][O][C][=Branch1][C][=O][/C][=C][/C][C][C@H1...,InChI=1S/C22H28O4/c1-21-9-8-15(23)10-13(21)4-6...,"methyl (2Z)-2-[(8S,9S,10R,13S,14S)-10,13-dimet...",2.5,60.4,5.8
1,1,53239731,Qwen/Qwen3-4B-Instruct-2507,IUPAC name,XLogP,CC(O)=N[C@@H]1[C@@H](O[C@@H]2O[C@@H](C)[C@@H](...,[C][C][Branch1][C][O][=N][C@@H1][C@@H1][Branch...,InChI=1S/C28H48N2O19/c1-7-15(34)19(38)21(40)27...,"N-[(2S,3R,4R,5S,6R)-2-[(2R,3S,4R,5R,6R)-5-acet...",-6.9,325.0,-1.5


process raw response

In [134]:
import pandas as pd

pd.to_numeric(
    raw_df["raw_responses"],
    errors="coerce"
).isna().mean()

np.float64(0.0248710949347892)

In [135]:
# TODO: 
raw_df["pred_values"] = pd.to_numeric(
    raw_df["raw_responses"],
    errors="coerce"
)

In [136]:
raw_df.to_json(f"data/exp/chem_property_pred/raw_responses/{output_file_suffix}.jsonl", orient="records", lines=True)

calculate metrics

In [137]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr

def cal_metrics(df, chem_property_colname: str):
    """
    Calculate various metrics for a given chemical property.
    chem_preperty: str, the name of a numeric property column in the dataframe, e.g., "xlogp" or "tpsa".
    """
    y_true = df[chem_property_colname].values
    y_pred = df["pred_values"].values
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    pearson_corr, _ = pearsonr(y_true, y_pred)
    rho, _ = spearmanr(y_true, y_pred)
    bias = (y_pred - y_true).mean()
    within_0_5 = (np.abs(y_pred - y_true) <= 0.5).mean()
    within_1 = (np.abs(y_pred - y_true) <= 1).mean()
    return pd.Series({
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "pearson_corr": pearson_corr,
        "spearman_corr": rho,
        "bias": bias,
        "within_0_5": within_0_5,
        "within_1": within_1,
    })

In [138]:
# dropna of gt_values or pred_values or other possible missing columns
clean_df = raw_df.dropna()

In [139]:
save_df = clean_df.groupby(["model_id", "rep_type", "tested_chem_property"]).apply(cal_metrics, chem_property_colname=chem_property_clean_colname).reset_index()
save_df

,model_id,rep_type,tested_chem_property,mae,mse,rmse,r2,pearson_corr,spearman_corr,bias,within_0_5,within_1
0,Qwen/Qwen3-4B-Instruct-2507,IUPAC name,XLogP,4.577012,46.484708,6.817969,-0.19344,0.289031,0.301416,-1.308314,0.084946,0.170876


In [140]:
save_df.to_csv(f"data/exp/chem_property_pred/metrics/{output_file_suffix}_metrics.csv", index=False)